[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/dirac/dirac.ipynb)

# The Dirac Electron in the Spacetime Algebra

The Dirac equation describes the electron: its mass, its charge and its spin of one half. Here the electron's state at a point is an even multivector of the spacetime algebra, and sandwiching vectors with it is a map on spacetime. That map carries the observer's frame to the electron's own frame: the time axis goes to the flow of the electron's charge, and the z axis goes to its spin. This notebook reads the electron's current, spin and energy off that map, finds the energies of plane waves, and follows what happens when positive and negative energy are mixed: the charge circles close to the speed of light, on a loop smaller than the electron's Compton wavelength.

Units are natural, ħ = c = 1, with the electron's mass as the unit of energy. Lengths are then in units of ħ/mc, 386 femtometres.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from itertools import accumulate

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import STA
from examples.relativity.dirac import render

np.set_printoptions(precision=3, suppress=True)

# Spacetime, with signature (+, -, -, -).
ga = STA
mv = NumpyContext(ga).multivector
Vector = ga.gatype.vector()
Bivector = ga.gatype.bivector()
Even = ga.gatype.even()
# Vectors in the observer's space, perpendicular to the time axis.
Spatial = ga.gatype.from_blades("x y z")
# The observer's time axis, gamma0, and the pseudoscalar.
time = mv.t                                     # [] Vector
I = mv.txyz                                     # [] Pseudoscalar
print(Even.output_subspace)

## 1. The spinor as a map on spacetime

The state of an electron at a point is an even multivector, eight real numbers. It factors as ψ = √ρ e^(Iβ/2) R, a density ρ, an angle β and a Lorentz rotor R. Sandwiching a vector with it, `psi >> Vector`, applies the rotor and scales by the density. The result is a map on spacetime that carries the observer's frame to the electron's. Its image of the time axis is the current, the flow of the electron's charge, and its image of the z axis is the spin. `psi.symmetric_reverse_product()`, ψψ̃, holds what is left over, ρ e^(Iβ).

In [ ]:
def spinor(density: float, beta: float, rotor: Even) -> Even:
    """The spinor with the given density, angle beta and Lorentz rotor."""
    return (I * (beta / 2)).exp() * rotor * np.sqrt(density)


# A boost along +z, and a turn about it; with it, a spinor of density 1.5 and beta 0.4.
rotor = ((mv.tz * -0.6 + mv.xy * 0.8) * 0.5).exp()     # [] Rotor
psi = spinor(1.5, 0.4, rotor)                           # [] Even
# rho times a Lorentz map.
frame = psi >> Vector                                   # [] Vector <- Vector
# The flow of charge, and the spin axis in units of hbar / 2.
current = frame(time)                                   # [] Vector
spin = frame(mv.z)                                      # [] Vector
# rho e^(I beta): a scalar and a pseudoscalar.
invariants = psi.symmetric_reverse_product()            # [] Even

In [ ]:
print("density, current . time: ", (current | time).to_array())
print("velocity:                ", render.relative((current ^ time) / (current | time)))
print("spin axis:               ", render.spatial(spin))
print("current squared:         ", (current | current).to_array(), "= rho squared")
print("rho cos(beta), rho sin(beta):", invariants.select[0].to_array(), -(invariants * I).select[0].to_array())

The current is timelike, with length ρ; its spatial part is the velocity the boost gives the electron. The spin is spacelike and perpendicular to the current.

For comparison, in matrix notation the state reads as a column of four complex numbers, and the current and spin as the bilinear covariants ψ̄γ^μψ and ψ̄γ^μγ⁵ψ, computed component by component from the gamma matrices.

## 2. The angle β

β does not appear in the current or the spin. The pseudoscalar anticommutes with every vector, so e^(Iβ/2) cancels in the sandwich. It does act on bivectors, the planes of spacetime. The frame's extension to bivectors, its outermorphism, turns a plane the way the rotor does and scales it by ρ². The spinor's own sandwich on bivectors does the same and also multiplies by e^(Iβ). Composing one with the inverse of the other leaves exactly that factor, divided by ρ.

In [ ]:
# The frame acting on planes, and the spinor acting on planes by its own sandwich.
extended = frame.outermorphism(Bivector)                # [] Bivector <- Bivector
own = psi >> Bivector                                   # [] Bivector <- Bivector
# Multiplication by e^(I beta) / rho, read off one plane.
difference = own(extended.inverse())                    # [] Bivector <- Bivector
plane = mv.xy                                           # [] Bivector
factor = difference(plane) * plane.inverse()            # [] Even

In [ ]:
print("difference on a plane, scalar and pseudoscalar:", factor.select[0].to_array(), -(factor * I).select[0].to_array())
print("cos(beta) / rho, sin(beta) / rho:              ", np.cos(0.4) / 1.5, np.sin(0.4) / 1.5)

For comparison, in matrix notation β is the Yvon–Takabayasi angle, the phase of ψ̄ψ + iψ̄γ⁵ψ.

## 3. Plane waves and the mass shell

For a plane wave of spatial momentum p the Dirac equation becomes a linear map on spinors, the Hamiltonian. The momentum acts as a relative vector, p γ₀, multiplying from the left, and the mass acts through the reflection in the time axis, γ₀ ψ γ₀. Right multiplication by the spin plane γ₂γ₁ is a map that squares to minus one and commutes with the Hamiltonian. The Hamiltonian's eigenvalues are ±√(p² + m²), each fourfold: two spin states, each paired with its image under that map.

In [ ]:
mass = 1.0


def hamiltonian(momentum: Vector):
    """The plane-wave Dirac equation as a map on spinors."""
    return (momentum * time) * Even + mass * (time * Even * time)             # [...] Even <- Even


# The spin plane gamma2 gamma1 = I sigma3, and right multiplication by it.
spin_plane = mv.yx                                                            # [] Bivector
imaginary = Even * spin_plane                                                 # [] Even <- Even
p = mv(Spatial, np.array([0.3, 0.0, 0.4]))                                    # [] Spatial
values, states = hamiltonian(p).eigh()                                        # [8] Scalar, [8] Even
# rho cos(beta) of each state.
signs = states.symmetric_reverse_product().select[0]                          # [8] Scalar

In [ ]:
print("energies:           ", values.to_array())
print("rho cos(beta):      ", signs.to_array())

Every positive-energy state has β = 0 and every negative-energy state β = π, where ψψ̃ is negative. In Hestenes' reading of the Dirac theory, β is what tells an electron from its antiparticle. Sweeping the momentum over a plane traces the two sheets of the mass shell, separated by a gap of twice the mass.

For comparison, in matrix notation the Hamiltonian reads as H = α·p + βm, with α and β 4×4 complex matrices, and right multiplication by the spin plane reads as multiplication by the imaginary unit i in iħ∂ψ/∂t.

In [ ]:
# Momenta sampled along each axis.
samples = 41
along = np.linspace(-2.0, 2.0, samples)
px, py = np.meshgrid(along, along)
momenta = mv(Spatial, np.stack([px, py, np.zeros_like(px)], axis=-1))        # [samples, samples] Spatial
shell, _ = hamiltonian(momenta).eigh()                                         # [samples, samples, 8] Scalar
render.draw_mass_shell(momenta, shell, mass);

## 4. Zitterbewegung

An electron with only positive energy moves steadily at its group velocity. Mixing in negative energy changes that. The positive part turns in the spin plane one way at the rate E, the negative part turns the other way, and where they overlap the current swings around at 2E. The charge circles in the plane perpendicular to the spin, on a loop of radius up to 1/(2E), about 190 femtometres for an electron at rest. Schrödinger found this motion in 1930 and called it Zitterbewegung, trembling motion. The negative part drifts against the momentum, so the more of it is mixed in, the slower the loop climbs, and at an even mix it does not climb at all.

In [ ]:
def evolve(momentum: Vector, psi: Even, times: np.ndarray) -> Even:
    """The plane wave's spinor over time: its positive- and negative-energy parts turn in the
    spin plane at the energy, in opposite senses."""
    E = (mass**2 - (momentum | momentum)).square_root()                       # [] Scalar
    # The positive-energy part, and e^(I sigma3 E t).
    positive = 0.5 * (psi + hamiltonian(momentum)(psi) / E)                  # [] Even
    turn = (spin_plane * E * times).exp()                                     # [times] Rotor
    return positive * turn.reverse() + (psi - positive) * turn


def path(psi: Even, dt: float) -> Bivector:
    """The positions the current carries a point through: its velocity, the relative vector over
    the density, summed step by step."""
    flow = psi >> time                                                        # [times] Vector
    # Relative vectors.
    velocities = (flow ^ time) / (flow | time)                                # [times] Bivector
    return stack(list(accumulate(velocities[:-1] * dt, initial=velocities[0] * 0.0)))


p = mv(Spatial, np.array([0.0, 0.0, 0.3]))                                    # [] Spatial
H, E = hamiltonian(p), (mass**2 - (p | p)).square_root()
# Positive energy with spin along z, and negative energy turning the other way.
electron = 0.5 * (mv.scalar([1.0]) + H(mv.scalar([1.0])) / E)              # [] Even
positron = 0.5 * (mv.tx - H(mv.tx) / E)                                      # [] Even
electron = electron / electron.symmetric_reverse_product().select[0].square_root()
positron = positron / (-positron.symmetric_reverse_product().select[0]).square_root()
times = np.linspace(0.0, 12.0, 600)
# The share of negative energy.
mixtures = (0.1, 0.25, 0.5)
spinors = [evolve(p, electron * np.sqrt(1 - share) + positron * np.sqrt(share), times) for share in mixtures]
paths = [path(psi, times[1] - times[0]) for psi in spinors]
render.draw_paths(times, spinors, paths, mixtures);

In [ ]:
# checks
# The current has length rho; beta is exactly what the outermorphism misses.
np.testing.assert_allclose((current | current).to_array(), 1.5**2, rtol=1e-8)
np.testing.assert_allclose((factor - (I * 0.4).exp() / 1.5).kernel, 0.0, atol=1e-9)
# The plane wave's energies are minus and plus E, fourfold, and right multiplication by the spin
# plane squares to minus one.
energy = np.sqrt(mass**2 + 0.3**2 + 0.4**2)
np.testing.assert_allclose(values.to_array(), [-energy] * 4 + [energy] * 4, rtol=1e-10)
probe = mv(Even.output_subspace, np.random.default_rng(0).normal(size=8))
np.testing.assert_allclose(imaginary(imaginary(probe)).kernel, -probe.kernel, atol=1e-12)
# An even mix does not drift along the momentum.
assert abs(render.relative(paths[-1])[:, 2]).max() < 1e-2